# Phase 2 — Workplace-level modeling table

**Grain:** 1 row = 1 `WORKPLACE ID`  
**Cohort (D008):** ≥1 visit in 2017–2022 AND ≥1 visit in 2023 (~12,234 workplaces)  
**Features:** visits/orders through **2022-12-31**  
**Target (D004):** ≥1 serious order in **2023** (Stop Work + Time Unknown)

**Leakage rules:**
1. Never use 2023+ rows when computing features.
2. `days_since_last_visit` uses last **2017–2022** visit only.
3. Do not include workplaces without a 2023 visit in the evaluation table (zeros = "not inspected," not "safe").
4. Do not stack 2024 yet (D007/D009).

In [13]:
from pathlib import Path
import pandas as pd
import numpy as np
ROOT = Path("..").resolve()
DATA = ROOT / "data"
OUT = ROOT / "data" / "processed"
OUT.mkdir(parents=True, exist_ok=True)
CORE_COLS = [
    "WORKPLACE ID",
    "ORDER TYPE",
    "CASE TYPE",
    "FIELD VISIT DATE",
    "PRIMARY NAICS",
    "ORDER STATUS",
]
SNAPSHOT = pd.Timestamp("2022-12-31") # used for the cutoff 
HIST_YEARS = range(2017, 2023)   # 2017..2022 inclusive
TARGET_YEAR = 2023
SERIOUS_2023 = {
    "Stop Use/Stop Work Order",
    "Time Unknown Order",
}
STOP_WORK_HIST = {"Stop Use/Stop Work Order"}
TIME_UNKNOWN_HIST = {"Time Unknown Order"}

def load_year(year, cols=None): 
    path = DATA / f"{year}_ohs_field_visit.xlsx"
    df = pd.read_excel(path)
    df["source_year"] = year
    return df 

def has_order(s):
    """Row has an enforcement action (not a visit-only row)."""
    return s.notna()

Now let's join all thetables from 2017-2022 Onwards. 

In [14]:
hist = pd.concat(
    [load_year(y, CORE_COLS) for y in HIST_YEARS],
    ignore_index=True,
)

hist["FIELD VISIT DATE"] = pd.to_datetime(hist["FIELD VISIT DATE"])

# Hard cutoff — anything later is leakage
hist = hist[hist["FIELD VISIT DATE"] <= SNAPSHOT].copy()
hist = hist.dropna(subset=["WORKPLACE ID"])
hist["WORKPLACE ID"] = hist["WORKPLACE ID"].astype(int)

print(f"History rows: {len(hist):,}")
print(f"History workplaces: {hist['WORKPLACE ID'].nunique():,}")
print(f"Date range: {hist['FIELD VISIT DATE'].min().date()} → {hist['FIELD VISIT DATE'].max().date()}")

History rows: 835,150
History workplaces: 169,092
Date range: 2017-01-02 → 2022-12-31


2023 Data will only be used for (a) cohort membership, (b) target- only and never for features

Now let's load the year and do some checks 

In [17]:
df23 = load_year(2023, CORE_COLS)
df23["FIELD VISIT DATE"] = pd.to_datetime(df23["FIELD VISIT DATE"])

df23 = df23[df23["FIELD VISIT DATE"] > pd.Timestamp("2022-12-31")
            & df23["FIELD VISIT DATE"] <= pd.Timestamp("2023-12-31")]

df23["WORKPLACE ID"] = df23["WORKPLACE ID"].astype(int)

print(f"2023 rows: {df23.shape[0]:,}")
print(f"2023 workplaces: {df23['WORKPLACE ID'].nunique():,}")

TypeError: unsupported operand type(s) for &: 'Timestamp' and 'DatetimeArray'